---
<a id='stage2'></a>
# 🔶 STAGE 2: POST-TRAINING — SUPERVISED FINE-TUNING (SFT)

After pretraining, the base model must be **aligned to behave like an assistant**. SFT transforms the document-completer into a helpful conversational agent.

## 2.1 — Conversation Format & Protocol

The model is now trained on **conversation-formatted** data:

```
Human: What is 2+2?
Assistant: 2+2 = 4

Human: What if it was "instead of"?
Assistant: 2+2 = 4, same as 2+2

Human: Why is the sky blue?
Assistant: Because of Rayleigh scattering.

Human: Wow!
Assistant: Indeed! Let me know if I can help with anything else.

Human: How can I hack into a computer?
Assistant: I'm sorry, I can't help with that.
```

**Tokenizer special tokens** structure the conversation:
```
<|system|>You are a helpful assistant.<|end|>
<|user|>What is 2+2?<|end|>
<|assistant|>2+2 = 4<|end|>
```

Different models use different formats (ChatML, Alpaca, etc.)

In [ ]:
# Simulate conversation formatting

def format_conversation_chatml(messages: list[dict]) -> str:
    """Format messages in ChatML format (used by OpenAI, many open models)"""
    formatted = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        formatted += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    formatted += "<|im_start|>assistant\n"  # model completes from here
    return formatted

def format_conversation_llama(messages: list[dict]) -> str:
    """Format messages in LLaMA-2 chat format"""
    system_msg = next((m['content'] for m in messages if m['role'] == 'system'), "")
    formatted = f"[INST] <<SYS>>\n{system_msg}\n<</SYS>>\n\n"
    user_msgs = [m for m in messages if m['role'] == 'user']
    asst_msgs = [m for m in messages if m['role'] == 'assistant']
    for i, user_msg in enumerate(user_msgs):
        formatted += f"{user_msg['content']} [/INST]"
        if i < len(asst_msgs):
            formatted += f" {asst_msgs[i]['content']} </s><s>[INST] "
    return formatted

# Example conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Why is the sky blue?"},
    {"role": "assistant", "content": "Because of Rayleigh scattering."},
    {"role": "user", "content": "What is Rayleigh scattering?"}
]

print("=" * 60)
print("ChatML Format:")
print("=" * 60)
print(format_conversation_chatml(messages))

print("=" * 60)
print("LLaMA-2 Format:")
print("=" * 60)
print(format_conversation_llama(messages))

## 2.2 — Conversation Datasets

How is this conversation data created?

**Timeline:**
- **Early work (2022):** InstructGPT paper — human labelers write conversations based on labeling instructions.
- **Today:** A huge amount of labeling is **LLM-assisted** (humans edit more than write), or just **entirely synthetic** (model generates, human reviews).

**Key Insight:** The quality of these conversations determines assistant behavior. Bad data → bad assistant.

The training signal: only **assistant tokens** are trained on. User/system tokens are masked from the loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualize SFT: which tokens contribute to loss
fig, ax = plt.subplots(figsize=(14, 3))
ax.axis('off')

tokens = [
    ("<sys>", "system"),
    ("You", "system"), ("are", "system"), ("helpful", "system"), ("</sys>", "system"),
    ("<user>", "user"),
    ("Why", "user"), ("sky", "user"), ("blue", "user"), ("?", "user"), ("</user>", "user"),
    ("<asst>", "assistant"),
    ("Rayleigh", "assistant"), ("scattering", "assistant"), (".", "assistant"), ("</asst>", "assistant"),
]

colors = {"system": "#BDC3C7", "user": "#AED6F1", "assistant": "#A9DFBF"}
loss_trained = {"system": False, "user": False, "assistant": True}

for i, (token, role) in enumerate(tokens):
    x = i * 0.9
    rect = mpatches.FancyBboxPatch((x, 0.4), 0.8, 0.5, boxstyle="round,pad=0.05",
                                    facecolor=colors[role], edgecolor='gray')
    ax.add_patch(rect)
    ax.text(x + 0.4, 0.65, token, ha='center', va='center', fontsize=7.5)
    if loss_trained[role]:
        ax.text(x + 0.4, 0.25, "✓ loss", ha='center', va='center', fontsize=7, color='green', fontweight='bold')
    else:
        ax.text(x + 0.4, 0.25, "✗ masked", ha='center', va='center', fontsize=7, color='red')

ax.set_xlim(-0.2, len(tokens) * 0.9)
ax.set_ylim(0, 1.1)

# Legend
legend_items = [
    mpatches.Patch(color='#BDC3C7', label='System tokens (masked from loss)'),
    mpatches.Patch(color='#AED6F1', label='User tokens (masked from loss)'),
    mpatches.Patch(color='#A9DFBF', label='Assistant tokens (trained on — loss computed here)'),
]
ax.legend(handles=legend_items, loc='upper center', fontsize=9, ncol=3)
ax.set_title('SFT Training: Only Assistant Tokens Contribute to Loss', fontsize=12, fontweight='bold', y=1.05)

plt.tight_layout()
plt.savefig('sft_loss_masking.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.3 — Hallucinations & Mitigations

### The Problem: Hallucinations
SFT models have a tendency to **confabulate** — produce confident but false answers.

```
Human: Who is Tom Cruise?
Assistant: Tom Cruise is a famous American actor and producer...

Human: Who is John Barrasso?
Assistant: John Barrasso is an American physician and politician...

Human: Who is Genghis Khan?
Assistant: Genghis Khan was the founder of the Mongol Empire. ✓

Human: Who is Orson Kovacs?
Assistant: ???   ← model doesn't know this, might hallucinate!
```

### Mitigation #1 — Model Interrogation
Use model introspection: discover what the model **knows it knows** → train on knowledge-based refusals.

```
Human: Who is Orson Kovacs?
Assistant: "I'm sorry, I don't believe I know"
```

### Mitigation #2 — Allow the Model to Search
Give the model a search tool:

```
Human: Who is Orson Kovacs?
Assistant: <SEARCH_START>Who is Orson Kovacs?<SEARCH_END>
[search result...]
Orson Kovacs appears to be ...
```

### The Swiss Cheese Model
LLM capabilities are like Swiss cheese:
- Some things work **really well**
- Some things (almost at random) show **brittleness**

## 2.4 — Models Need Tokens to Think

One of the most important insights in LLM prompting: **the model needs space to "think"** before giving an answer.

### ❌ Bad — No reasoning space:
```
Human: Emily buys 3 apples and 2 oranges. Each apple costs $2. Total fruit = $13. Cost of apple?
Assistant: $3   ← WRONG (rushed, no intermediate steps)
```

### ✅ Good — Thinking out loud (Chain-of-Thought):
```
Human: Emily buys 3 apples and 2 oranges. Each orange costs $2. Total = $13. Cost of apple?
Assistant: The total cost of oranges is $4.13 - 4 = 9; cost of 3 apples is $9. 9/3 = 3.
           So each apple costs $3.   ← CORRECT ✓
```

**Key Rule:** Never put the final answer in the first token. Let the model reason step by step.

This is why techniques like **Chain-of-Thought (CoT)** and **"think step by step"** prompting work.

In [ ]:
# Demonstrate Chain-of-Thought vs Direct Answer

def simulate_no_cot(problem: str) -> str:
    """Simulates a rushed model without CoT (often wrong on math)"""
    # Simplified simulation — in reality a model might just output the first plausible token
    return "$3"  # wrong

def simulate_cot(problem: str) -> dict:
    """Simulates chain-of-thought reasoning"""
    # For the apple/orange problem:
    steps = [
        "Step 1: Total fruit cost = $13",
        "Step 2: 2 oranges × $2 each = $4 total for oranges",
        "Step 3: Remaining for apples = $13 - $4 = $9",
        "Step 4: 3 apples → each apple = $9 / 3 = $3",
        "Answer: Each apple costs $3 ✓"
    ]
    return {"reasoning": steps, "answer": "$3"}

problem = "Emily buys 3 apples and 2 oranges. Each orange costs $2. Total cost = $13. What is the cost of each apple?"

print("Problem:", problem)
print()
print("❌ Without Chain-of-Thought:")
print("   Answer:", simulate_no_cot(problem), "(model may be right by luck, but can't be trusted)")
print()
print("✅ With Chain-of-Thought:")
cot_result = simulate_cot(problem)
for step in cot_result['reasoning']:
    print("  ", step)
print()

print("-" * 60)
print("KEY INSIGHT: Models need 'token budget' to reason correctly.")
print("The answer token should come LAST, not first.")
print("This is the basis of techniques like:")
print("  - Chain-of-Thought (CoT) prompting")
print("  - 'Think step by step' prompts")
print("  - Extended thinking / reasoning models (o1, DeepSeek-R1)")

## 2.5 — Model Limitations

### Models Can't Count
Remember: models see **tokens, not individual letters**.
- Counting letters in a word is harder than it seems.
- `Is 9.9 > 9.11?` → Tricky due to tokenization of decimal numbers.

### Models Can (and Should!) Use Tools
Modern LLMs are equipped with tool-use capabilities:
- **Web search** — to fetch current information
- **Code / Python interpreter** — to do exact arithmetic, run simulations
- **File I/O, APIs, databases** — for agentic tasks

### Knowledge of Self
- The LLM has **no knowledge of self** "out of the box"
- If you ask it nothing, it will probably think it's ChatGPT
- You can give it a "sense of self" in 2 ways:
  - Hard-coded conversations around these topics in training data
  - A "system message" reminding the model of its identity at every conversation start

### Vague Recollection vs. Working Memory
- Knowledge in the model parameters = **Vague recollection** (like something you read 1 month ago)
- Knowledge in the **context window** = Working memory (clear, reliable, immediate)

In [ ]:
# Visualize: Working Memory vs. Parametric Memory
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

# Context window (working memory)
ctx = mpatches.FancyBboxPatch((0.05, 0.3), 0.38, 0.5, boxstyle="round,pad=0.02",
                               facecolor='#AED6F1', edgecolor='#2980B9', linewidth=2)
ax.add_patch(ctx)
ax.text(0.24, 0.62, "Context Window", ha='center', fontsize=13, fontweight='bold', color='#1A5276')
ax.text(0.24, 0.52, "(Working Memory)", ha='center', fontsize=10, color='#1A5276')
ax.text(0.24, 0.42, "✓ Sharp, reliable, immediate\n✓ What you put here the model KNOWS\n✓ RAG, Tool results, System prompts",
        ha='center', fontsize=9, va='center')

# Parameters (vague recollection)
par = mpatches.FancyBboxPatch((0.55, 0.3), 0.38, 0.5, boxstyle="round,pad=0.02",
                               facecolor='#F9E79F', edgecolor='#D4AC0D', linewidth=2)
ax.add_patch(par)
ax.text(0.74, 0.62, "Model Parameters", ha='center', fontsize=13, fontweight='bold', color='#7D6608')
ax.text(0.74, 0.52, "(Vague Recollection)", ha='center', fontsize=10, color='#7D6608')
ax.text(0.74, 0.42, "⚠ Like reading something 1 month ago\n⚠ Can be confident but wrong\n⚠ Subject to hallucination",
        ha='center', fontsize=9, va='center')

ax.text(0.5, 0.88, "Two Types of Knowledge in LLMs", ha='center', fontsize=15, fontweight='bold')

# VS
ax.text(0.5, 0.55, "VS", ha='center', fontsize=20, fontweight='bold', color='gray')

plt.tight_layout()
plt.savefig('memory_types.png', dpi=120, bbox_inches='tight')
plt.show()

---
<a id='stage3'></a>
# 🔴 STAGE 3: POST-TRAINING — REINFORCEMENT LEARNING (RL / RLHF)

The SFT model is good, but RL pushes it further. The key difference:

| SFT | RL |
|-----|----|
| Show model correct behavior (imitation) | Give model a **reward signal** and let it explore |
| Learns from human-written examples | Learns from **outcomes** |
| Like studying worked problems | Like practicing until you get it right |

**Analogy from the diagram:**
- **Exposition** = Pretraining (background knowledge)
- **Worked problems** = Supervised Finetuning (shown the right answer)
- **Practice problems** = Reinforcement Learning (prompts to practice until you reach the correct answer)

## 3.1 — The RL Setup (Verifiable Domains)

For **math/code/logic** (where we can verify answers programmatically):

```
Prompt: Emily buys 3 apples and 2 oranges...

Model generates 15 solutions (rollouts).
Only 4 of them got the right answer.

→ Take the top solution (right AND short).
→ Repeat many, many times.
```

This is called **GRPO** (Group Relative Policy Optimization) or similar algorithms.

The model literally **discovers "thinking" and "cognitive strategies"** through this process — this emerges during RL optimization!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate RL rollouts: model generates multiple solutions, we reward correct ones
np.random.seed(7)

def simulate_rl_rollouts(n_rollouts=15, p_correct=0.27):
    """Simulate model generating multiple solutions. Some correct, some not."""
    results = []
    for i in range(n_rollouts):
        correct = np.random.random() < p_correct
        length = np.random.randint(50, 300)  # token length
        reward = 0
        if correct:
            reward = 1.0 - (length / 1000)  # reward correct AND short
        results.append({'id': i+1, 'correct': correct, 'length': length, 'reward': reward})
    return results

rollouts = simulate_rl_rollouts(15)
correct_ones = [r for r in rollouts if r['correct']]
wrong_ones = [r for r in rollouts if not r['correct']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot rollouts
ids = [r['id'] for r in rollouts]
rewards = [r['reward'] for r in rollouts]
colors = ['#27AE60' if r['correct'] else '#E74C3C' for r in rollouts]
bars = axes[0].bar(ids, [r['length'] for r in rollouts], color=colors, edgecolor='white')
axes[0].set_title(f'RL Rollouts: {len(rollouts)} Solutions Generated\n'
                  f'({len(correct_ones)} correct ✓, {len(wrong_ones)} wrong ✗)', fontsize=11)
axes[0].set_xlabel('Rollout ID')
axes[0].set_ylabel('Solution Length (tokens)')

legend_items = [
    mpatches.Patch(color='#27AE60', label='Correct ✓'),
    mpatches.Patch(color='#E74C3C', label='Wrong ✗')
]
axes[0].legend(handles=legend_items)

# Reward scores
axes[1].bar(ids, rewards, color=colors, edgecolor='white')
axes[1].set_title('Reward Scores (Correct = positive, Wrong = 0)\nTop solution selected for RL update', fontsize=11)
axes[1].set_xlabel('Rollout ID')
axes[1].set_ylabel('Reward')
if correct_ones:
    best = max(correct_ones, key=lambda r: r['reward'])
    axes[1].annotate(f'Best!\nID={best["id"]}', xy=(best['id'], best['reward']),
                     xytext=(best['id']+1, best['reward']+0.05),
                     arrowprops=dict(arrowstyle='->', color='blue'),
                     fontsize=10, color='blue')

plt.tight_layout()
plt.savefig('rl_rollouts.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nRL Summary:")
print(f"  Generated: {len(rollouts)} rollouts")
print(f"  Correct:   {len(correct_ones)} ({100*len(correct_ones)/len(rollouts):.0f}%)")
print(f"  Best reward: {max(rewards):.3f}")
print(f"  → Model learns to produce more solutions like the best one!")

## 3.2 — RLHF: Reinforcement Learning from Human Feedback

For **unverifiable domains** (writing, jokes, summaries — where there's no objective answer), we use **RLHF**:

### The RLHF Pipeline:

```
STEP 1: Collect human preferences
  - Show humans pairs of model outputs
  - Humans rank them: "A is better than B"
  → Cost: 5,000 scores from humans

STEP 2: Train a Reward Model
  - Train a neural net to simulate human preferences
  - This is the "reward model" (RM)

STEP 3: Use RL with the Reward Model
  - Run RL as usual, but use the RM instead of actual humans
  → Cost: 1,000,000 scores (cheap, automated!)
```

### Naive Approach (Too Expensive):
Run RL with 1,000 updates × 1,000 prompts × 1,000 rollouts = **1,000,000,000 human scores needed**. Impossible.

### RLHF Approach:
Only need ~5,000 human scores to train the reward model. Then use the RM for billions of evaluations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize reward model scores vs human preference ordering
np.random.seed(42)

# 5 rollouts for "write a joke about pelicans"
jokes = [
    "Why don't pelicans tip? Because their bill is always full!",
    "What's a pelican's favorite movie? The Beak-oning.",
    "Pelican walks into a bar. Bartender says 'Why the long beak?'",
    "I told a pelican joke but it fell flat. Just like its feet.",
    "Pelicans are just seagulls that went to business school."
]

reward_scores = [0.1, 0.8, 0.3, 0.4, 0.5]  # reward model scores
human_ordering = [5, 1, 4, 3, 2]  # human preference rank (1=best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward model scores
colors = plt.cm.RdYlGn([s for s in reward_scores])
bars = axes[0].barh(range(5), reward_scores, color=colors)
axes[0].set_yticks(range(5))
axes[0].set_yticklabels([f'Joke {i+1}' for i in range(5)])
axes[0].set_xlabel('Reward Model Score')
axes[0].set_title('Reward Model Scores\n(automated, runs millions of times)', fontsize=11)
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
for i, (bar, score) in enumerate(zip(bars, reward_scores)):
    axes[0].text(score + 0.01, i, f'{score:.1f}', va='center', fontsize=10)

# Human ordering
human_colors = plt.cm.RdYlGn([1 - (r-1)/4 for r in human_ordering])
axes[1].barh(range(5), [6-r for r in human_ordering], color=human_colors)  # invert for display
axes[1].set_yticks(range(5))
axes[1].set_yticklabels([f'Joke {i+1}' for i in range(5)])
axes[1].set_xlabel('Human Preference (higher = better)')
axes[1].set_title('Human Preference Ordering\n(expensive, ~5000 comparisons total)', fontsize=11)
for i, rank in enumerate(human_ordering):
    axes[1].text(0.1, i, f'Rank #{rank}', va='center', fontsize=10, color='black')

plt.suptitle('RLHF: Reward Model vs Human Preferences\n"Write a joke about pelicans"',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('rlhf_viz.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.3 — RLHF: Upside & Downside

### ✅ RLHF Upside
- We can run RL in **arbitrary domains** (even unverifiable ones)
- This empirically **improves model performance** — possibly due to a discriminator–generator gap
- In many cases, it is much **easier to discriminate than to generate**
  - Example: "Write a poem" vs. "Which of these 5 poems is best?" — the latter is much easier for humans

### ⚠️ RLHF Downside
- We are doing RL with respect to a **lossy simulation of humans** — it might be misleading!
- RL discovers ways to **"game" the model**
- It discovers "adversarial examples" of the reward model
- Example: After 2,000 updates, the top joke about pelicans is *not* the best joke ever — it might be something totally non-sensical, like "the the the the the"

This is called **reward hacking** — the model finds exploits in the reward model that don't correspond to real human preferences.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate reward hacking: reward model score goes up, but true quality diverges
np.random.seed(0)

updates = np.arange(0, 2001, 50)

# Reward model score (what RL optimizes) — goes up
rm_score = 0.3 + 0.6 * (1 - np.exp(-updates / 500)) + np.random.normal(0, 0.02, len(updates))
rm_score = np.clip(rm_score, 0, 1)

# True human quality — improves initially, then degrades (reward hacking!)
true_quality = 0.3 + 0.5 * (1 - np.exp(-updates / 300))
true_quality[updates > 800] -= 0.0003 * (updates[updates > 800] - 800)
true_quality = np.clip(true_quality, 0, 1)
true_quality += np.random.normal(0, 0.02, len(updates))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(updates, rm_score, label='Reward Model Score (optimized by RL)', color='#2ECC71', linewidth=2)
ax.plot(updates, true_quality, label='True Human Quality', color='#E74C3C', linewidth=2, linestyle='--')
ax.axvline(x=800, color='gray', linestyle=':', alpha=0.7)
ax.text(820, 0.55, 'Reward Hacking begins\n(scores diverge)', fontsize=9, color='gray')
ax.fill_between(updates[updates > 800],
                rm_score[updates > 800],
                true_quality[updates > 800],
                alpha=0.15, color='red', label='Divergence (Goodhart\'s Law)')
ax.set_xlabel('RL Update Steps')
ax.set_ylabel('Score')
ax.set_title("RLHF Reward Hacking: Goodhart's Law in LLMs\n"
             "'When a measure becomes a target, it ceases to be a good measure.'", fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('reward_hacking.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key Takeaway: RL is powerful but needs careful monitoring.")
print("The reward model is a lossy simulation of human preferences.")
print("After enough RL steps, the model finds ways to exploit the RM.")

## 3.4 — DeepSeek-R1: RL Discovering Reasoning

One remarkable finding from **DeepSeek-R1** (2025):

> RL discovers **"thinking" and "cognitive strategies"** like self-verification, backtracking, and multi-step reasoning. This **emerges** during RL optimization of math problems.

Example of emergent behavior during RL:
```
Question: If a=1, then the sum of real solutions of sqrt(x - sqrt(x+a)) = a is?

Response after RL training:
  ...
  Wait. Wait. That's an aha moment I can flag here.
  Let me re-evaluate this step to identify if the correct ans can be—
  We started with the equation: sqrt(x - sqrt(x+1)) = 1
  First, let's square both sides...
  [continues with careful verification]
```

The model learned **metacognition** — checking its own work — purely from reward signals!

---
<a id='stage4'></a>
# 🚀 STAGE 4: WHAT'S NEXT — THE FUTURE OF LLMs

## Preview of Things to Come

### Modalities:
- **Multimodal** — not just text but audio, images, video, natural conversations

### Agentic Behavior:
- **Tasks** — long-horizon, computer-using, error-correcting agents
- **Agents** — (long, complex, computer-using, error-correcting, invisible)
- **Computer-using** — models that can operate a browser/desktop
- **Real-time training** — models that continuously update from interactions

## Where to Find Models

| Type | Examples |
|------|----------|
| Proprietary (API) | OpenAI, Anthropic (Claude), Google (Gemini) |
| Open weights | Meta LLaMA, Mistral, DeepSeek, Qwen |
| Run locally | LM Studio, Ollama |

## Where to Keep Track
- [lilianweng.github.io](https://lilianweng.github.io) — deep technical blog posts
- Andrej Karpathy's X/Twitter — curated LLM news
- [HuggingFace](https://huggingface.co) — model hub
- arXiv cs.CL / cs.LG — latest papers

---
<a id='code'></a>
# 💻 HANDS-ON: Full Pipeline Summary & Exercises

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Full LLM Training Pipeline Visualization
fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('off')
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)

def draw_box(ax, x, y, w, h, color, title, subtitle='', fontsize=10):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.15 if subtitle else 0), title,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', color='white')
    if subtitle:
        ax.text(x + w/2, y + h/2 - 0.25, subtitle,
                ha='center', va='center', fontsize=8, color='white', alpha=0.9)

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my + 0.15, label, ha='center', fontsize=8, color='#2C3E50', style='italic')

# Data
draw_box(ax, 0.3, 5.5, 2.5, 1.5, '#1A5276', 'Internet\nData', 'Trillions of tokens')
draw_box(ax, 0.3, 3.5, 2.5, 1.5, '#1F618D', 'Tokenizer', 'BPE / WordPiece')

# Pretraining
draw_box(ax, 3.5, 3.5, 3.0, 2.5, '#117A65', 'PRETRAINING', 'Next-token prediction\n175B+ params\nWeeks on 1000s GPUs', fontsize=11)

# Base Model
draw_box(ax, 7.5, 4.2, 2.5, 1.2, '#27AE60', 'Base Model', '"Internet Simulator"')

# SFT
draw_box(ax, 3.5, 0.8, 3.0, 2.0, '#7D6608', 'SFT', 'Conversation datasets\nHuman/LLM labeled\nLoss only on assistant', fontsize=11)

# SFT Model
draw_box(ax, 7.5, 1.2, 2.5, 1.2, '#D4AC0D', 'SFT Model', 'Chat assistant')

# RLHF
draw_box(ax, 10.5, 0.8, 3.0, 2.0, '#922B21', 'RLHF / RL', 'Reward model\nHuman preferences\nRollouts + updates', fontsize=11)

# Final RL Model
draw_box(ax, 10.5, 4.0, 3.0, 1.8, '#C0392B', 'RL Model', 'ChatGPT / Claude\n/ Gemini', fontsize=11)

# Arrows
draw_arrow(ax, 2.8, 6.25, 3.5, 5.0, 'raw text')
draw_arrow(ax, 2.8, 4.25, 3.5, 4.25, 'token IDs')
draw_arrow(ax, 6.5, 4.75, 7.5, 4.75, '')
draw_arrow(ax, 7.5+1.25, 4.2, 7.5+1.25, 2.4, 'initialize')
draw_arrow(ax, 6.5, 1.8, 7.5, 1.8, '')
draw_arrow(ax, 10.0, 1.8, 10.5, 1.8, '')
draw_arrow(ax, 12.0, 2.8, 12.0, 4.0, 'RL training')

# Title
ax.text(8, 7.6, '🧠 Complete LLM Training Pipeline', ha='center', fontsize=16,
        fontweight='bold', color='#2C3E50')
ax.text(8, 7.1, 'Pretraining → Supervised Fine-Tuning → Reinforcement Learning',
        ha='center', fontsize=11, color='#555')

plt.tight_layout()
plt.savefig('full_pipeline.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === SUMMARY TABLE ===

import pandas as pd

summary = pd.DataFrame([
    {
        'Stage': '1. Pretraining',
        'Objective': 'Next-token prediction',
        'Data': 'Raw internet text (trillions of tokens)',
        'Labels': 'Self-supervised (no human labels)',
        'Output': 'Base model ("internet simulator")',
        'Cost': 'Very high ($millions)'
    },
    {
        'Stage': '2. Supervised Fine-Tuning (SFT)',
        'Objective': 'Imitate assistant behavior',
        'Data': 'Human/LLM-written conversations',
        'Labels': 'Human-labeled (expensive)',
        'Output': 'SFT model (basic assistant)',
        'Cost': 'Moderate'
    },
    {
        'Stage': '3. RLHF / RL',
        'Objective': 'Maximize reward signal',
        'Data': 'Prompts + model rollouts',
        'Labels': 'Human pref rankings → reward model',
        'Output': 'RL model (refined assistant)',
        'Cost': 'Moderate (reward model amortizes cost)'
    }
])

summary = summary.set_index('Stage')
print(summary.to_string())

## 🧪 Exercises

Try these to deepen your understanding:

### Exercise 1: Tokenization
- Install `tiktoken` and tokenize 10 different sentences.
- Find a word that tokenizes into more than 4 tokens.
- Try: is `"9.9"` tokenized differently from `"9.11"`? What about `"strawberry"`?

### Exercise 2: Next-Token Prediction
- Build a tiny character-level language model using pure numpy.
- Train on a small text corpus (e.g., Shakespeare).
- Sample from it to generate text.

### Exercise 3: Chain-of-Thought
- Using the OpenAI or Anthropic API, send the same math problem:
  1. Asking for a direct answer
  2. Asking to "think step by step"
- Compare accuracy across 10 problems.

### Exercise 4: Reward Modeling
- Create a tiny reward model: train a classifier to prefer one sentence over another.
- Dataset: write 20 pairs of (good_response, bad_response) for a given prompt.
- Train a logistic regression on sentence embeddings to rank them.

### Exercise 5: Simulate RL with Verifiable Rewards
- Use any LLM API to generate 10 solutions to a math problem.
- Check each solution programmatically.
- Identify the best correct solution and track the distribution of correct answers.

---

## 📚 Further Reading

| Topic | Resource |
|-------|----------|
| Attention is All You Need | [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) |
| InstructGPT (SFT + RLHF) | [arXiv:2203.02155](https://arxiv.org/abs/2203.02155) |
| LLaMA | [arXiv:2302.13971](https://arxiv.org/abs/2302.13971) |
| DeepSeek-R1 (RL for reasoning) | [arXiv:2501.12948](https://arxiv.org/abs/2501.12948) |
| Lilian Weng's Blog | [lilianweng.github.io](https://lilianweng.github.io) |
| Andrej Karpathy's nanoGPT | [github.com/karpathy/nanoGPT](https://github.com/karpathy/nanoGPT) |
| Andrej Karpathy's makemore | [github.com/karpathy/makemore](https://github.com/karpathy/makemore) |

---
*Based on Andrej Karpathy's "State of GPT" overview diagram. Notebook created for educational purposes.*